In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from neural_spd.config import PLOTS_PATH, IS_HEADLESS, DATA_PATH, MODEL_DEM_DIR, TOPO_DERIVATIVES, NOISE_LEVELS

PLOTS_PATH.mkdir(parents=True, exist_ok=True)

# Pick a sample file for validation
sample_files = list((DATA_PATH / "0" / MODEL_DEM_DIR).glob("*.npy"))
sample_file = sample_files[0]
print(f"Validating with sample file: {sample_file.name}")

# Data types to validate (DEM + derivatives)
data_types = [MODEL_DEM_DIR] + list(TOPO_DERIVATIVES.keys())

# Create plot: rows = data types, columns = noise levels
fig, axes = plt.subplots(len(data_types), len(NOISE_LEVELS), figsize=(6*len(NOISE_LEVELS), 4*len(data_types)))
fig.suptitle('Data Processing Validation: Noise Levels vs Data Types', fontsize=16)

# Handle case where there's only one noise level or one data type
if len(NOISE_LEVELS) == 1:
    axes = axes.reshape(-1, 1)
if len(data_types) == 1:
    axes = axes.reshape(1, -1)

for row_idx, data_type in enumerate(data_types):
    for col_idx, noise_level in enumerate(NOISE_LEVELS):
        noise_str = str(noise_level).replace('.', '-')
        data_file = DATA_PATH / noise_str / data_type / sample_file.name

        if data_file.exists():
            data = np.load(data_file)

            # Choose colormap based on data type
            if 'flow_accumulation' in data_type:
                cmap = 'Blues'
            elif data_type == 'slope':
                cmap = 'Reds'
            elif data_type == 'curvature':
                cmap = 'RdBu'
            elif data_type == MODEL_DEM_DIR:
                cmap = 'terrain'
            else:
                cmap = 'viridis'

            # Plot the data
            im = axes[row_idx, col_idx].imshow(data, cmap=cmap)
            axes[row_idx, col_idx].axis('off')

            # Calculate differences if this isn't the first noise level
            if col_idx > 0:
                # Compare with noise level 0
                clean_file = DATA_PATH / "0" / data_type / sample_file.name
                if clean_file.exists():
                    clean_data = np.load(clean_file)
                    diff = data - clean_data

                    mean_diff = np.mean(diff)
                    min_diff = np.min(diff)
                    max_diff = np.max(diff)

                    title = f"{data_type}\nNoise={noise_level}\nΔ: μ={mean_diff:.3f}\nmin={min_diff:.3f}, max={max_diff:.3f}"
                else:
                    title = f"{data_type}\nNoise={noise_level}\n(Clean file missing)"
            else:
                title = f"{data_type}\nNoise={noise_level}\n(Reference)"

            axes[row_idx, col_idx].set_title(title, fontsize=10)

            # Add colorbar
            plt.colorbar(im, ax=axes[row_idx, col_idx], shrink=0.8, aspect=20)

        else:
            axes[row_idx, col_idx].text(0.5, 0.5, f'{data_type}\nNoise={noise_level}\nFILE MISSING', 
                                      ha='center', va='center', transform=axes[row_idx, col_idx].transAxes,
                                      bbox=dict(boxstyle="round,pad=0.3", facecolor="red", alpha=0.3))
            axes[row_idx, col_idx].axis('off')

plt.tight_layout()
if not IS_HEADLESS:
    plt.show()
plt.savefig(PLOTS_PATH / "noise_validation.png")

print("✓ Validation plot complete!")
print("Check that:")
print("  - DEMs look like realistic topography") 
print("  - Slopes look like slope (bright = steep)")
print("  - Curvature shows ridges/valleys (red/blue)")
print("  - Flow accumulation shows drainage networks")
print("  - Noise versions show non-zero differences")
